In [ ]:
def train_risk_models(df_transactions):
    global SCORING_TEMPLATES_AUTO

    df_transactions['date'] = pd.to_datetime(df_transactions['date'], errors='coerce')
    df_transactions['description'] = df_transactions['description'].fillna('unknown').str.lower().str.strip()
    df_transactions['merchantName'] = df_transactions['merchantName'].fillna('unknown').str.lower().str.strip()
    df_transactions['type'] = df_transactions['type'].fillna('unknown').str.lower().str.strip()

    if 'predicted_subcategory' not in df_transactions.columns:
        print("⚡ 'predicted_subcategory' missing, using 'subcategory' as fallback...")
        df_transactions['predicted_subcategory'] = df_transactions['subcategory'].fillna('unknown').str.lower().str.strip()
    else:
        df_transactions['predicted_subcategory'] = df_transactions['predicted_subcategory'].fillna('unknown').str.lower().str.strip()

    df_transactions = smart_detect_recurrence(df_transactions)
    df_features = feature_engineering(df_transactions)

    user_info = df_transactions.groupby("userId").agg({
        "user_profile": "first",
        "defaulted": "first"
    }).reset_index()

    df_train = df_features.merge(user_info, on="userId", how="inner")
    print(f"\n✅ Feature engineered {df_train.shape[0]} users.")

    if not os.path.exists("risk_models"):
        os.makedirs("risk_models")

    profiles = df_train["user_profile"].unique()

    for profile in profiles:
        print(f"\n=== Training models for profile: {profile} ===")
        df_profile = df_train[df_train["user_profile"] == profile]

        X = df_profile.drop(columns=["userId", "user_profile", "defaulted"])
        y = df_profile["defaulted"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        param_grid = {
            'reg_alpha': [0, 0.01, 0.1, 1],
            'reg_lambda': [0.1, 1, 5, 10],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.1]
        }

        model = LGBMClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        )

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        search = RandomizedSearchCV(
            model,
            param_distributions=param_grid,
            n_iter=20,
            scoring='f1',
            cv=skf,
            verbose=1,
            n_jobs=-1,
            random_state=42
        )

        search.fit(X_train, y_train)
        best_model = search.best_estimator_

        preds = best_model.predict(X_test)
        y_prob = best_model.predict_proba(X_test)[:, 1]

        f1 = f1_score(y_test, preds)
        acc = accuracy_score(y_test, preds)
        auc = roc_auc_score(y_test, y_prob)

        print(f"Final Metrics | Test F1 Score: {f1:.2f} | Test Accuracy: {acc:.2f} | Test AUC: {auc:.2f}")

        model_filename = f"risk_models/{profile.lower()}_risk_model.pkl"
        joblib.dump(best_model, model_filename)
        print(f"✅ Saved model: {model_filename}")

        importances = best_model.feature_importances_
        feature_names = X.columns

        total_importance = np.sum(importances)
        feature_weights = {}

        for feature, imp in zip(feature_names, importances):
            normalized = imp / total_importance if total_importance > 0 else 0
            if normalized > 0.01:
                feature_weights[feature] = normalized

        SCORING_TEMPLATES_AUTO[profile.lower()] = feature_weights

    joblib.dump(SCORING_TEMPLATES_AUTO, "risk_models/auto_scoring_templates.pkl")
    print("\n✅ Auto scoring templates saved.")
    print("\n🏁 All profile models trained with hyperparameter tuning and scoring templates generated.")

